# Phase 2: Hybrid Model Training Pipeline (v2 - Persistent Auto-Resume)
This notebook contains the production training logic for the hybrid weapon detector, now enhanced with an **auto-resume system** to handle environment restarts in Lightning AI Studio.

### 1. Imports and Environment Setup

In [ ]:
import os
import sys
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from pathlib import Path

# --- Project Root Discovery ---
def get_project_root():
    """Traverses up from the current file to find the project root (identified by requirements.txt)."""
    current_path = Path().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / "requirements.txt").exists():
            return parent
    return current_path

PROJECT_ROOT = get_project_root()
os.chdir(PROJECT_ROOT)  # Sync CWD to root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"✅ Environment Aligned. Project Root: {PROJECT_ROOT}")
# ----------------------------------------------

# Project imports 
from models.hybrid_model import HybridWeaponDetector
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import DEFAULT_CFG


### 2. Dataset Alignment

In [ ]:
def get_dataloaders(data_yaml_path, batch_size=16, imgsz=640):
    """Initialises real YOLO dataloaders for the hybrid model."""
    data_cfg = check_det_dataset(data_yaml_path)
    
    train_set = YOLODataset(
        img_path=data_cfg['train'],
        imgsz=imgsz,
        augment=True,
        batch_size=batch_size,
        task='detect',
        data=data_cfg
    )
    
    val_set = YOLODataset(
        img_path=data_cfg['val'],
        imgsz=imgsz,
        augment=False,
        batch_size=batch_size,
        task='detect',
        data=data_cfg
    )
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, collate_fn=train_set.collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, collate_fn=val_set.collate_fn)
    
    return train_loader, val_loader

### 3. Production Training Loop with Checkpointing

In [ ]:
class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda", checkpoint_path="checkpoint.pth"):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.checkpoint_path = checkpoint_path
        
        # Loss function (using our custom Focal Head's loss method)
        self.criterion = model.head.compute_loss

    def save_checkpoint(self, optimizer, epoch):
        """Saves a checkpoint, overwriting the previous one to save space (50GB limit)."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        # Use a temporary file and rename to ensure atomicity
        temp_path = self.checkpoint_path + ".tmp"
        torch.save(checkpoint, temp_path)
        os.replace(temp_path, self.checkpoint_path)
        print(f"--> Checkpoint saved to {self.checkpoint_path} at epoch {epoch}")

    def load_checkpoint(self):
        """Loads the checkpoint if it exists."""
        if os.path.exists(self.checkpoint_path):
            print(f"--> Found existing checkpoint at {self.checkpoint_path}. Loading state...")
            checkpoint = torch.load(self.checkpoint_path, map_location=self.device)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            return checkpoint['epoch'], checkpoint['optimizer_state_dict']
        return 0, None
        
    def train_epoch(self, optimizer, epoch):
        self.model.train()
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        total_loss = 0
        
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            
            optimizer.zero_grad()
            
            # Forward pass
            preds = self.model(imgs)
            
            # Compute loss (matches ground truth to anchors)
            loss = self.criterion(preds, batch, self.device)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        return total_loss / len(self.train_loader)

    def run(self, epochs=50):
        # 1. Check for existing checkpoints
        start_epoch, optimizer_state = self.load_checkpoint()
        
        # 2. Initialise Model State and Optimizer
        # If resuming, we need to ensure the backbone freeze state matches the epoch
        if start_epoch < 10:
            print("[INFO] Initialising with frozen backbone...")
            self.model.backbone.freeze()
            optimizer = optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=1e-4)
        else:
            print("[INFO] Resuming with unfrozen backbone fine-tuning...")
            self.model.backbone.unfreeze()
            optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)

        # 3. Load Optimizer State if available
        if optimizer_state:
            optimizer.load_state_dict(optimizer_state)
            print(f"--> Resuming training from epoch {start_epoch + 1}")
        
        # 4. Main Training Loop (starts from last_completed_epoch + 1)
        for epoch in range(start_epoch + 1, epochs + 1):
            # Unfreeze backbone at epoch 10
            if epoch == 10:
                print("\n[INFO] Unfreezing backbone for fine-tuning...")
                self.model.backbone.unfreeze()
                optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)
            
            avg_loss = self.train_epoch(optimizer, epoch)
            print(f"Epoch {epoch} Average Loss: {avg_loss:.4f}")
            
            # Periodic Saving: Auto-resume checkpoint (overwrites to save space)
            self.save_checkpoint(optimizer, epoch)
            
            # Milestone weights (every 5 epochs)
            if epoch % 5 == 0:
                self.model.save(f"models/weights/epoch_{epoch}.pt")
        
        self.model.save("models/weights/best.pt")

### 4. Execution
Configured to use persistent storage if available.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device}")

# --- Persistence Configuration ---
# Lightning AI Studio persistent storage path
PERSISTENT_DIR = "/teamspace/studios/this_studio/"
if not os.path.exists(PERSISTENT_DIR):
    PERSISTENT_DIR = str(PROJECT_ROOT)

CHECKPOINT_PATH = os.path.join(PERSISTENT_DIR, "last_state.pth")
WEIGHTS_DIR = os.path.join(PROJECT_ROOT, "models/weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f"Persistent storage: {PERSISTENT_DIR}")
print(f"Checkpoint file: {CHECKPOINT_PATH}")
# ----------------------------------

# 1. Model Setup
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", pretrained=True, nc=3, device=device)

# 2. Data Setup
data_yaml = r"data/processed/yolo_dataset/data.yaml"
if os.path.exists(data_yaml):
    train_loader, val_loader = get_dataloaders(data_yaml, batch_size=8)
    
    # 3. Training Initiation
    # Using larger epoch count for production, trainer will handle resume
    trainer = HybridTrainer(model, train_loader, val_loader, device=device, checkpoint_path=CHECKPOINT_PATH)
    
    # To start/resume training:
    trainer.run(epochs=50)
    
    print("\n[SUCCESS] Pipeline aligned. Ready for production training.")
else:
    print(f"[ERROR] data.yaml not found at {data_yaml}. Run setup_data script first.")